In [1]:
import matplotlib.pyplot as plt
from nilearn.connectome import ConnectivityMeasure
from brainspace.gradient import GradientMaps
from brainspace.utils.parcellation import map_to_labels
import numpy as np
import nibabel as nib
from nilearn import datasets
import os.path as op
import os
from nilearn import signal
import pandas as pd

bids_folder = '/Users/mrenke/data/ds-stressrisk'
bids_folder_ = '/Volumes/mrenkeED/data/ds-stressrisk'

subList = ['10','11','12','13','14'] #['01','02','03','04','05','08','09',]
ses = 1
specification=''

In [2]:
atlas = datasets.fetch_atlas_surf_destrieux()
regions = atlas['labels'].copy()
masked_regions = [b'Medial_wall', b'Unknown']
masked_labels = [regions.index(r) for r in masked_regions]
for r in masked_regions:
    regions.remove(r)

# Build Destrieux parcellation and mask
labeling = np.concatenate([atlas['map_left'], atlas['map_right']])
labeling_noParcel = np.arange(0,len(labeling),1,dtype = int)     # Map gradients to original parcels


In [8]:
from utils import cleanTS
from scipy.sparse.csgraph import connected_components

target_folder = op.join(bids_folder_,'derivatives','correlation_matrices')

for sub in subList:
    mask = ~np.isin(labeling, masked_labels)
    clean_ts = cleanTS(sub, ses,bids_folder=bids_folder)
    seed_ts = clean_ts[mask]

    # filter out nodes that are not connected to the rest
    correlation_measure = ConnectivityMeasure(kind='correlation')
    graph = correlation_measure.fit_transform([seed_ts.T])[0] #correlation_matrix_noParcel
    pd.DataFrame(graph).to_csv(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_unfiltered.csv')) # .shape = (18709, 18709)
    corr_matrices_unfiltered.append(pd.DataFrame(graph))

    cc = connected_components(graph)
    mask_cc = cc[1] == 0 # all nodes in 0 belong to the largest connected component, check #-components in cc[0]
    mask[mask == True] = mask_cc
    seed_ts = clean_ts[mask]   

    correlation_measure = ConnectivityMeasure(kind='correlation')
    correlation_matrix = correlation_measure.fit_transform([seed_ts.T])[0]
    pd.DataFrame(correlation_matrix).to_csv(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_onlyconncomp.csv')) # .shape = (18709, 18709)
    
    print(sub, 'done')
    

    
   

10 done


In [4]:
# one single subjects
from utils import cleanTS
from scipy.sparse.csgraph import connected_components

sub = '09'

target_folder = op.join(bids_folder_,'derivatives','correlation_matrices')

corr_matrices_unfiltered = []
corr_matrices = []


mask = ~np.isin(labeling, masked_labels)
clean_ts = cleanTS(sub, ses,bids_folder=bids_folder)
seed_ts = clean_ts[mask]

# filter out nodes that are not connected to the rest
correlation_measure = ConnectivityMeasure(kind='correlation')
graph = correlation_measure.fit_transform([seed_ts.T])[0] #correlation_matrix_noParcel
pd.DataFrame(graph).to_csv(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_unfiltered.csv')) # .shape = (18709, 18709)
#corr_matrices_unfiltered.append(pd.DataFrame(correlation_matrix))

cc = connected_components(graph)
mask_cc = cc[1] == 0 # all nodes in 0 belong to the largest connected component, check #-components in cc[0]
mask[mask == True] = mask_cc
seed_ts = clean_ts[mask]   

correlation_measure = ConnectivityMeasure(kind='correlation')
correlation_matrix = correlation_measure.fit_transform([seed_ts.T])[0]
pd.DataFrame(correlation_matrix).to_csv(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_onlyconncomp.csv')) # .shape = (18709, 18709)

#corr_matrices.append(pd.DataFrame(correlation_matrix))
print(sub, 'done')

09 done


In [ ]:
# now load in matrices and compute average

corr_matrices = []
for sub in subList:
    cm = pd.read_csv(op.join(target_folder,f'sub-{sub}_ses-{ses}_corrMatrix_onlyconncomp.csv'))
    corr_matrices.append(cm)
